# D3QN V4 — Offline Root-Cause Audit (không re-encode)

**Mục tiêu:** Kiểm tra nguyên nhân Q-value/terminal học chưa tốt từ **checkpoint và replay đã huấn luyện**, trước khi chỉnh source V5.

Notebook thực hiện: (1) xác thực SAFE ZIP V4 và khôi phục **single-step reward / realized Monte Carlo return** từ n-step=3; (2) kiểm tra **Double-DQN TD target, Huber saturation, reward spikes** theo coding-step; (3) **supervised remaining-return prediction** với cùng kiến trúc Dueling Q network, chia **video-disjoint Train/Validation/Holdout**; (4) so sánh **action ranking Best–Latest trên đúng cùng observations**; (5) xuất báo cáo CSV/JSON và biểu đồ.

**Không dùng** VVenC, VVdeC, YOLO, tập BDD100K hay Test30. **Không training RL**; chỉ huấn luyện hai mô hình supervised nhỏ trên replay có sẵn. GPU là tùy chọn, Kaggle CPU vẫn dùng được.

⚠️ **Lưu ý khoa học:** Realized MC return được sinh bởi behavior policy có exploration, **không phải** nhãn optimal Q(s,a); supervised fit tốt không tự động chứng minh Bellman bug. Coding-step index không nhất thiết tương ứng POC hiển thị hay intra-frame.

**Chuẩn bị:** Tải `D3QN_V4_TERMINAL_REPLAY_MINI16_VAL4_SAFE.zip` từ lần chạy trước, thêm ZIP này vào Kaggle → **Add Input** (có thể tạo private Kaggle Dataset để đính kèm). Bật **Internet ON** để tải mã audit từ GitHub; **GPU không bắt buộc**. Sau đó **Run All**.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, zipfile, shutil
from IPython.display import display, Image, FileLink

# Chỉ sửa biến này nếu Kaggle không tự tìm đúng SAFE ZIP.
SAFE_ZIP = None
ANALYSIS_COMMIT = "3967c583c038fbda6052c6532bc9a4207fdad969"
ANALYSIS_BRANCH = "analysis/d3qn-v4-offline-root-cause"
REPO = Path("/kaggle/working/rl_vvc_v4_offline_source")
OUT = Path("/kaggle/working/D3QN_V4_OFFLINE_ROOT_CAUSE")
RESULT_ZIP = Path("/kaggle/working/D3QN_V4_OFFLINE_ROOT_CAUSE_RESULTS.zip")
SUPERVISED_EPOCHS = 35  # Không phải RL training

# Kaggle có thể tự giải nén SAFE ZIP thành thư mục dataset.
# SAFE_ZIP=None: tự tìm ZIP hoặc thư mục đã giải nén.
# Có thể chỉ định SAFE_ZIP đến file .zip hoặc đường dẫn thư mục.
REQUIRED_SAFE = {
    "learning_manifest.json", "train_history.json",
    "checkpoints/latest_checkpoint.pt", "checkpoints/best_checkpoint.pt",
}

def _valid_manifest(value):
    return (value.get("protocol") == "native-masked-d3qn-qp/v4-terminal-aware"
            and value.get("source_commit") == "ace1db10e05b140ad49e1a676f80563219fb1ede"
            and value.get("status") == "complete"
            and value.get("completed_steps") == 5120
            and value.get("test_used") is False)

def _valid_extracted(folder):
    if not folder.is_dir() or not all((folder / x).is_file() for x in REQUIRED_SAFE):
        return False
    try:
        return _valid_manifest(json.loads(
            (folder / "learning_manifest.json").read_text(encoding="utf-8")))
    except (OSError, ValueError, UnicodeError):
        return False

def _valid_zip(path):
    if not path.is_file() or not zipfile.is_zipfile(path):
        return False
    try:
        with zipfile.ZipFile(path) as z:
            return (REQUIRED_SAFE <= set(z.namelist()) and
                    _valid_manifest(json.loads(z.read("learning_manifest.json"))))
    except (OSError, ValueError, zipfile.BadZipFile):
        return False

if SAFE_ZIP is None:
    choices = set()
    for root in (Path("/kaggle/input"), Path("/kaggle/working")):
        if not root.is_dir():
            continue
        for file in root.rglob("D3QN_V4_TERMINAL_REPLAY_MINI16_VAL4_SAFE.zip"):
            if _valid_zip(file):
                choices.add(file.resolve())
        for file in root.rglob("learning_manifest.json"):
            if _valid_extracted(file.parent):
                choices.add(file.parent.resolve())
    choices = sorted(choices, key=str)
    if len(choices) != 1:
        print("SAFE_INPUT_CANDIDATES", [str(p) for p in choices])
        print("KAGGLE_INPUT_DIRECTORIES",
              [str(p) for p in Path("/kaggle/input").iterdir()]
              if Path("/kaggle/input").is_dir() else [])
        raise FileNotFoundError(
            "Không tìm thấy đúng 1 bộ SAFE V4. Nếu Kaggle tự giải nén, "
            "đặt SAFE_ZIP tới thư mục chứa learning_manifest.json và checkpoints/. "
            "Hoặc chọn file .zip nguyên vẹn.")
    source = choices[0]
else:
    source = Path(SAFE_ZIP).resolve()

if source.is_dir():
    if not _valid_extracted(source):
        raise ValueError(f"Thư mục SAFE V4 thiếu file hoặc sai nguồn: {source}")
    # Offline audit đọc ZipFile; đóng gói lại đúng các file đã giải nén.
    # Không thay đổi bất kỳ checkpoint, replay hoặc reward nào.
    SAFE_ZIP = Path("/kaggle/working/D3QN_V4_SAFE_REPACKED_FOR_AUDIT.zip")
    allowed = {
        "learning_manifest.json", "train_history.json", "baseline.json",
        "monitor_history.json", "value_audit.json",
        "checkpoints/latest_checkpoint.pt", "checkpoints/best_checkpoint.pt",
    }
    paths = [source / name for name in sorted(allowed) if (source / name).is_file()]
    trace_dir = source / "frame_traces"
    if trace_dir.is_dir():
        paths.extend(sorted(trace_dir.glob("*.jsonl.gz")))
    with zipfile.ZipFile(SAFE_ZIP, "w", compression=zipfile.ZIP_STORED) as z:
        for file in paths:
            if file.is_symlink():
                raise ValueError(f"Không chấp nhận symlink: {file}")
            z.write(file, arcname=file.relative_to(source).as_posix())
    print("KAGGLE_AUTO_EXTRACT_DETECTED", source)
    print("REPACKED_CHECKPOINTS_AND_TRACES", len(paths), "files")
else:
    if not _valid_zip(source):
        raise ValueError(f"Không phải SAFE V4 ZIP hợp lệ: {source}")
    SAFE_ZIP = source
    print("ORIGINAL_SAFE_ZIP_DETECTED", SAFE_ZIP)

assert SAFE_ZIP.is_file(), f"Không tìm thấy {SAFE_ZIP}"
with zipfile.ZipFile(SAFE_ZIP) as z:
    required = {"learning_manifest.json","train_history.json",
                "checkpoints/latest_checkpoint.pt","checkpoints/best_checkpoint.pt"}
    assert required <= set(z.namelist()), "Thiếu checkpoint hoặc history trong SAFE ZIP"
    manifest = json.loads(z.read("learning_manifest.json"))
assert manifest["status"] == "complete"
assert manifest["completed_steps"] == 5120
assert manifest["test_used"] is False
assert manifest["source_commit"] == "ace1db10e05b140ad49e1a676f80563219fb1ede"
print("SAFE_ZIP_OK", SAFE_ZIP)
print("TRAIN_STEPS", manifest["completed_steps"], "BEST", manifest["best_steps"],
      "BEST_J", manifest["best_objective"])
print("No BDD100K files, video codec or GPU required.")


In [ ]:
GIT_URL = "https://github.com/youngan2810/RL-VVC.git"
if not (REPO / ".git").exists():
    subprocess.run(["git","clone","--branch",ANALYSIS_BRANCH,"--depth","1",GIT_URL,str(REPO)],check=True)
subprocess.run(["git","fetch","--no-tags","--depth","1","origin",ANALYSIS_COMMIT],
               cwd=REPO,check=True)
subprocess.run(["git","checkout","--detach",ANALYSIS_COMMIT],cwd=REPO,check=True)
sha = subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO,text=True).strip()
assert sha == ANALYSIS_COMMIT
assert (REPO/"scripts/rl_rc_dot/d3qn_v4_offline_audit.py").is_file()
print("OFFLINE_ANALYSIS_SOURCE_PIN_PASS",sha)


In [ ]:
OUT.mkdir(parents=True,exist_ok=True)
cmd = [sys.executable,"-u","-m","scripts.rl_rc_dot.d3qn_v4_offline_audit",
       "--safe-zip",str(SAFE_ZIP),"--out",str(OUT),
       "--epochs",str(SUPERVISED_EPOCHS),"--device","auto"]
print("RUNNING OFFLINE AUDIT (NO video encoding)",flush=True)
print(" ".join(cmd),flush=True)
subprocess.run(cmd,cwd=REPO,check=True)
assert (OUT/"audit_summary.json").exists()
assert (OUT/"supervised_report.json").exists()
print("OFFLINE_ANALYSIS_COMPLETE")


In [ ]:
summary=json.loads((OUT/"audit_summary.json").read_text(encoding="utf-8"))
sup=json.loads((OUT/"supervised_report.json").read_text(encoding="utf-8"))
print("=== V4 CHECKPOINT DIAGNOSTICS ===")
for label, row in summary["checkpoint_diagnostics"].items():
    print(f"{label:6} steps={row['steps']:4d}  terminal_abs_TD={row['terminal_td_abs_mean']:.5f}"
          f"  Huber_saturation={100*row['huber_fraction_abs_td_gt_1']:.1f}%"
          f"  behavior_MC_minus_Q={row['mean_behavior_mc_minus_q']:.5f}")
print("\n=== SAME-STATE ACTION RANK ===")
print("Greedy agreement Best vs Latest:",
      f"{100*summary['same_state_ranking']['greedy_agreement_fraction']:.2f}%")
print("\n=== SUPERVISED VIDEO-HOLDOUT ===")
for name, score in sup["models"].items():
    print(f"{name:26} RMSE={score['rmse']:.4f}  MAE={score['mae']:.4f}"
          f"  R2={score['r2']:.4f}")
print("\nSố video trong mỗi split:",
      {k:len(v) for k,v in sup["split"].items()})
for name in ("reward_td_by_coding_step.png",
             "supervised_validation_curve.png","supervised_holdout_scatter.png"):
    path=OUT/name
    if path.is_file():
        print("\n",name)
        display(Image(filename=str(path)))


In [ ]:
(OUT/"analysis_runner_provenance.json").write_text(json.dumps({
    "analysis_commit":ANALYSIS_COMMIT,
    "v4_training_source_commit":"ace1db10e05b140ad49e1a676f80563219fb1ede",
    "input_safe_zip":SAFE_ZIP.name,
    "new_rl_training":False,
    "reencoding":False,
    "note":"Behavior Monte Carlo is not optimal Q ground truth."
},indent=2),encoding="utf-8")

with zipfile.ZipFile(RESULT_ZIP,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUT.iterdir()):
        if path.is_file():
            z.write(path,arcname=path.name)
assert RESULT_ZIP.is_file()
print("RESULTS_ZIP",RESULT_ZIP,"SIZE_MB",
      round(RESULT_ZIP.stat().st_size/(1024**2),3))
display(FileLink(str(RESULT_ZIP)))


## Đọc kết quả và gửi lại để quyết định V5

Tải **`D3QN_V4_OFFLINE_ROOT_CAUSE_RESULTS.zip`** ở cell cuối, rồi gửi lại file ZIP (hoặc tối thiểu `audit_summary.json`, `supervised_report.json`, `td_reward_groups.csv`, `same_state_action_ranking.json`).

- Nếu supervised network học được remaining return **trên holdout video chưa thấy khi train** nhưng TD vẫn không khớp, ưu tiên kiểm tra Bellman targets, bootstrapping và horizon. Đây chỉ là tín hiệu, không tự động chứng minh lỗi TD.
- Nếu supervised thua baseline progress/rate hoặc R² holdout kém, cần xem lại state, network và số lượng dữ liệu. Không nên kết luận vấn đề chắc chắn nằm ở một yếu tố duy nhất.
- `validation_reward_components.csv` (nếu có compact traces trong SAFE) đối chiếu đúng ba thành phần F1, bit và rate-debt; `td_reward_groups.csv` chỉ rõ nhóm coding-step có sai lệch lớn; `same_state_action_ranking.json` xem checkpoint Best/Latest có đảo thứ hạng action trên **cùng** observation hay không.

**Bảo đảm không có video mới được mã hóa, không sửa checkpoint V4 và không mở Test30.**
